<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-fp/blob/main/goit_rdb_fp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Relational Databases — Final Project

## Task 1. Database Setup and Data Import

In [15]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

In [16]:
%pip install -q "psycopg[binary]" sqlalchemy pandas pyarrow requests

In [17]:
import os
import sys
import pandas as pd
import pgserver

from urllib.request import urlretrieve
from sqlalchemy import create_engine, text
from google.colab import files

print("Python:", sys.version)

pg = pgserver.get_server('/tmp/goit_rdb_fp', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [18]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS pandemic;
        SET search_path TO pandemic;
    """))

    result = conn.execute(text("SHOW search_path;")).scalar_one()
    print("Current schema search path:", result)

Current schema search path: pandemic


In [19]:
#uploaded = files.upload() -> download file from local folder

df = pd.read_csv("infectious_cases.csv")

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())

Dataset shape: (10521, 12)


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,NaN,6160.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1981,NaN,5859.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1982,NaN,9730.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1983,NaN,13937.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1984,NaN,3864.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10521 entries, 0 to 10520
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Entity                10521 non-null  object 
 1   Code                  9393 non-null   object 
 2   Year                  10521 non-null  int64  
 3   Number_yaws           74 non-null     float64
 4   polio_cases           7610 non-null   float64
 5   cases_guinea_worm     8428 non-null   float64
 6   Number_rabies         6780 non-null   float64
 7   Number_malaria        6780 non-null   float64
 8   Number_hiv            6780 non-null   float64
 9   Number_tuberculosis   6780 non-null   float64
 10  Number_smallpox       104 non-null    float64
 11  Number_cholera_cases  2967 non-null   float64
dtypes: float64(9), int64(1), object(2)
memory usage: 986.5+ KB


None

In [20]:
df.to_sql(
    name="infectious_cases",
    con=engine,
    schema="pandemic",
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("CSV imported successfully.")

CSV imported successfully.


In [21]:
display(pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT "Entity") AS unique_entities,
        MIN("Year") AS min_year,
        MAX("Year") AS max_year,
        COUNT(*) FILTER (
            WHERE "Number_rabies" IS NULL
        ) AS missing_rabies
    FROM pandemic.infectious_cases;
""", engine))

,total_rows,unique_entities,min_year,max_year,missing_rabies
0,10521,245,1920,2023,3741


## Task 2. Database Normalization to Third Normal Form (3NF)

Separate unique entity-code combinations from yearly infectious disease statistics.